In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

print("GROQ_API_KEY:", bool(os.getenv("GROQ_API_KEY")))

GROQ_API_KEY: True


In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

## RAG implementation with your OWN text data ##

#### STEP 1: Prepare Document for you text ####

In [15]:
from langchain_core.documents import Document # Document is a LangChain data structure used to store page content and metadata, now this list contains only one Document - len(docs) => 1

my_text = """In computer science, in particular in knowledge representation and reasoning and metalogic, the area of automated reasoning is dedicated to understanding different aspects of reasoning. The study of automated reasoning helps produce computer programs that allow computers to reason completely, or nearly completely, automatically. Although automated reasoning is considered a sub-field of artificial intelligence, it also has connections with theoretical computer science and philosophy.

The most developed subareas of automated reasoning are automated theorem proving (and the less automated but more pragmatic subfield of interactive theorem proving) and automated proof checking (viewed as guaranteed correct reasoning under fixed assumptions).[citation needed] Extensive work has also been done in reasoning by analogy using induction and abduction.[1]

Other important topics include reasoning under uncertainty and non-monotonic reasoning. An important part of the uncertainty field is that of argumentation, where further constraints of minimality and consistency are applied on top of the more standard automated deduction. John Pollock's OSCAR system is an example of an automated argumentation system that is more specific than being just an automated theorem prover.

Tools and techniques of automated reasoning include the classical logics and calculi, fuzzy logic, Bayesian inference, reasoning with maximal entropy and many less formal ad hoc techniques.

In the 2020s, to enhance the ability of large language models to solve complex problems, AI researchers have designed reasoning language models that can spend additional time on the problem before generating an answer[2] and neuro-symbolic architectures that use symbolic reasoning systems to prevent hallucinations.[3][4][5]

Early years
The development of formal logic played a big role in the field of automated reasoning, which itself led to the development of artificial intelligence. A formal proof is a proof in which every logical inference has been checked back to the fundamental axioms of mathematics. All the intermediate logical steps are supplied, without exception. No appeal is made to intuition, even if the translation from intuition to logic is routine. Thus, a formal proof is less intuitive and less susceptible to logical errors.[6]

Some consider the Cornell Summer meeting of 1957, which brought together many logicians and computer scientists, as the origin of automated reasoning, or automated deduction.[7] Others say that it began before that with the 1955 Logic Theorist program of Newell, Shaw and Simon, or with Martin Davis’ 1954 implementation of Presburger's decision procedure (which proved that the sum of two even numbers is even).[8]

Automated reasoning, although a significant and popular area of research, went through an "AI winter" in the eighties and early nineties. The field subsequently revived, however. For example, in 2005, Microsoft started using verification technology in many of their internal projects and is planning to include a logical specification and checking language in their 2012 version of Visual C.[7]

Significant contributions
Principia Mathematica was a milestone work in formal logic written by Alfred North Whitehead and Bertrand Russell. Its purpose was to derive all or some of the mathematical expressions, in terms of symbolic logic. Principia Mathematica was initially published in three volumes in 1910, 1912 and 1913.[9] It succeeded The Principles of Mathematics, a 1903 book by Bertrand Russell, in which Russell had presented his famous paradox and argued his thesis that mathematics and logic are identical.

Logic Theorist (LT) was the first ever program developed in 1956 by Allen Newell, Cliff Shaw and Herbert A. Simon to "mimic human reasoning" in proving theorems and was demonstrated on fifty-two theorems from chapter two of Principia Mathematica, proving thirty-eight of them.[10] In addition to proving the theorems, the program found a proof for one of the theorems that was more elegant than the one provided by Whitehead and Russell. After an unsuccessful attempt at publishing their results, Newell, Shaw, and Herbert reported in their publication in 1958, The Next Advance in Operation Research:

"There are now in the world machines that think, that learn and that create. Moreover, their ability to do these things is going to increase rapidly until (in a visible future) the range of problems they can handle will be co- extensive with the range to which the human mind has been applied."[11]"""

docs = [Document(page_content=my_text, metadata={"source":"kulavilakkamman text", "DocumentId":"01"})] # With a PDF, you use a PDF loader to extract the text and create Document objects automatically.

#### STEP 2: Splitting the document into CHUNKS ####

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # Maximum size of each chunk is around 500 characters
    chunk_overlap=50 # 50 characters are shared between consecutive chunks (since, a sentence can be broken into chunk1 and rest into chunk2)
)

chunks = splitter.split_documents(docs) # multiple Document created from the single Document

#### STEP 3: Create Embeddings for these CHUNKS

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings 

embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2") 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13032.08it/s]


#### STEP 4: Create Embeddings and Store in ChromaDB ####

In [ ]:
from langchain_chroma import Chroma # created embeddings are stored in RAM, since we did not persist or did not provided cloud server crendentials

vector_store = Chroma.from_documents( # langchain creates embeddings for those chunks and stores it in chromadb
    documents=chunks,
    embedding=embedding_model
)

#### STEP 5: Semantic Search, pick TOP K vectors ####

In [ ]:
context = vector_store.similarity_search("What is Computer Science", k=3) # returns array of Document(chunk_id, chunk_text, metadata)

#### STEP 6: Create Updated prompt, with the retrieved Context ####

In [20]:
llm.invoke(f"What is Computer Science? You can answer using following context: {context}")

AIMessage(content='Computer science is the systematic study of computation, information processing, and the design of algorithms and systems that can solve problems automatically.  At its core it blends mathematics, logic, and engineering to understand how problems can be represented, reasoned about, and solved by machines.\n\nA key area that illustrates this blend is **automated reasoning**—the field that seeks to give computers the ability to reason “completely, or nearly completely, automatically.”  Automated reasoning sits at the intersection of:\n\n* **Knowledge representation and reasoning** – how to encode facts, rules, and relationships so that a computer can manipulate them.\n* **Metalogic** – the study of the logical foundations that underpin reasoning systems.\n* **Artificial intelligence** – where automated reasoning is a sub‑field, but it also draws on and contributes to **theoretical computer science** and even **philosophy**.\n\nHistorically, automated reasoning has expe